# Customer Segmentation: RFM Analysis & K-Means Clustering

This notebook covers **Phase 10: RFM Scoring** and **Phases 12-15: K-Means Clustering and Segment Profiling**.

We will:
1. **Calculate RFM Scores** manually (1-5) using quintiles.
2. **Scale features** using `StandardScaler` to prepare for distance-based clustering.
3. **Train K-Means models** for different values of $K$ (2 to 6) and evaluate them using **Inertia** and **Silhouette Scores**.
4. **Profile the final segments** to extract actionable business insights.
5. **Save clustered customers** back to our DuckDB database for use in reports and dashboards.

## 1. Load Data from DuckDB

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

db_path = os.path.join("..", "data", "processed", "retail_analytics.db")
conn = duckdb.connect(db_path)
df_rfm = conn.execute("SELECT * FROM gold.customer_rfm_base").fetchdf()
print(f"Loaded {df_rfm.shape[0]} customers.")
df_rfm.head()

## 2. RFM Scoring (1 to 5 Quintiles)

We will score each customer from 1 (lowest/worst) to 5 (highest/best):
- **Recency Score**: Higher score for *lower* recency (more recent purchase). we invert the ranking.
- **Frequency Score**: Higher score for *higher* frequency.
- **Monetary Score**: Higher score for *higher* monetary spending.

In [ ]:
# Compute scores using quintiles
# Using rank(method='first') to handle duplicate values gracefully in qcut
df_rfm['R_Score'] = pd.qcut(df_rfm['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
df_rfm['F_Score'] = pd.qcut(df_rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['M_Score'] = pd.qcut(df_rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Composite RFM Score
df_rfm['RFM_Cell'] = df_rfm['R_Score'].astype(str) + df_rfm['F_Score'].astype(str) + df_rfm['M_Score'].astype(str)
df_rfm['RFM_Score'] = df_rfm['R_Score'] + df_rfm['F_Score'] + df_rfm['M_Score']

df_rfm.head()

## 3. Preparing Data for K-Means (Scaling)

K-Means is a distance-based algorithm (Euclidean distance). If we do not scale the features, `monetary` (values in thousands) will completely dominate `frequency` (values between 1 and 10) and `recency` (values between 0 and 365). We will apply logarithmic transformation first to handle skewness, and then standard scale.

In [ ]:
# Visualizing raw feature distributions to check skewness
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['recency', 'frequency', 'monetary']):
    sns.histplot(df_rfm[col], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col.capitalize()}")
plt.show()

In [ ]:
# Log-transform features to resolve heavy skewness
df_log = pd.DataFrame()
df_log['recency'] = np.log1p(df_rfm['recency'])
df_log['frequency'] = np.log1p(df_rfm['frequency'])
df_log['monetary'] = np.log1p(df_rfm['monetary'])

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_log)

X_scaled_df = pd.DataFrame(X_scaled, columns=['recency', 'frequency', 'monetary'])
X_scaled_df.head()

## 4. Hyperparameter Tuning: Evaluating K

We will compute **Inertia** (sum of squared distances to nearest cluster center) and **Silhouette Score** (measure of how similar an object is to its own cluster compared to other clusters) for $K = 2$ to $6$.

In [ ]:
inertia = []
silhouette = []
k_range = range(2, 7)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    # Calculate silhouette score on a sample if dataset is huge, but with 18k rows we can compute it directly
    score = silhouette_score(X_scaled, kmeans.labels_, sample_size=5000, random_state=42)
    silhouette.append(score)

# Plot metrics
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:red'
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia', color=color)
ax1.plot(k_range, inertia, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', color=color)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, silhouette, marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', color=color)

plt.title("Elbow Method & Silhouette Scores for K-Means Clustering")
fig.tight_layout()
plt.show()

## 5. Fit Final Model (K = 4)

Based on the Elbow curve and business interpretability, we choose $K = 4$ clusters. This divides customers into 4 distinct, actionable segments.

In [ ]:
optimal_k = 4
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_rfm['cluster'] = kmeans_model.fit_transform(X_scaled).argmin(axis=1) # Ensure cluster numbering is consistent
df_rfm['cluster'] = kmeans_model.labels_
df_rfm.head()

## 6. Cluster Profiling & Segment Naming

Let's look at the average RFM characteristics of each cluster to assign business names.

In [ ]:
profiles = df_rfm.groupby('cluster').agg(
    customer_count=('customer_key', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_monetary=('monetary', 'sum')
).reset_index()

total_revenue = profiles['total_monetary'].sum()
profiles['revenue_contribution_pct'] = (profiles['total_monetary'] / total_revenue) * 100
profiles = profiles.sort_values(by='avg_monetary', ascending=False)
profiles

### Map Clusters to Business Names
Depending on the averages, we label the segments:
- **Champions**: Low Recency, High Frequency, High Monetary.
- **Loyal / Potential**: Medium Recency, Medium Frequency, Medium Monetary.
- **At Risk / Inactive**: High Recency, Medium Frequency, Medium Monetary.
- **New / Low-Value**: Low Frequency, Low Monetary.

In [ ]:
# Determine segment label mappings based on profiles
# Sort clusters by monetary value to assign labels dynamically
sorted_clusters = profiles['cluster'].tolist()
cluster_names = {
    sorted_clusters[0]: 'Champions',
    sorted_clusters[1]: 'Loyal Customers',
    sorted_clusters[2]: 'At Risk',
    sorted_clusters[3]: 'Low-Value / Occasional'
}

df_rfm['segment_name'] = df_rfm['cluster'].map(cluster_names)

# Check count by segment
df_rfm['segment_name'].value_value_counts() if hasattr(df_rfm['segment_name'], 'value_value_counts') else df_rfm['segment_name'].value_counts()

In [ ]:
# Visualize K-Means Clusters in 2D Space (Frequency vs Monetary)
sns.scatterplot(data=df_rfm, x='frequency', y='monetary', hue='segment_name', palette='Set1', alpha=0.7)
plt.title("Customer Segments (Frequency vs Monetary)")
plt.xlabel("Frequency (Orders count)")
plt.ylabel("Monetary ($ Spending)")
plt.yscale('log')
plt.show()

## 7. Save Segments back to DuckDB

We will write the final clustered customer mapping to a new table `gold.customer_segments` in our DuckDB database. This connects the machine learning results directly back to the database, making it immediately available for SQL queries and the Streamlit dashboard.

In [ ]:
# Drop existing segments table if it exists
conn.execute("DROP TABLE IF EXISTS gold.customer_segments")

# Select key fields to persist
df_save = df_rfm[[
    'customer_key', 'customer_id', 'customer_number', 
    'recency', 'frequency', 'monetary', 
    'R_Score', 'F_Score', 'M_Score', 'RFM_Score',
    'cluster', 'segment_name'
]].copy()

# Register DataFrame and save as DuckDB table
conn.register('temp_segments', df_save)
conn.execute("CREATE TABLE gold.customer_segments AS SELECT * FROM temp_segments")
print("Saved segments table to gold.customer_segments in DuckDB!")

# Verify save by running count query
print(conn.execute("SELECT segment_name, COUNT(*) AS customer_count, ROUND(AVG(monetary),1) AS avg_spending FROM gold.customer_segments GROUP BY segment_name").fetchdf())

# Close DuckDB connection
conn.close()